In [ ]:
import pandas as pd
import boto3
from botocore.exceptions import ClientError
from langchain_core.prompts import PromptTemplate
import os
from dotenv import load_dotenv
import anthropic
from datetime import datetime, timedelta, timezone
from boto3.dynamodb.conditions import Key
from dateutil.relativedelta import relativedelta
from datetime import date
import json

In [35]:
def load_api_key():
    # override=True にすることで、.env のセットアップが既存の環境変数を上書きします
    env_path = os.path.join(os.getcwd(), "../.env")
    load_dotenv(dotenv_path=env_path, override=True)
    os.environ["ANTHROPIC_API_KEY"] = os.getenv("Anthropic_API_Key")

def get_dynamo_data(table_name,user_id,index_name=None,days=None,region_name='ap-northeast-1'):
    dynamodb = boto3.resource('dynamodb', region_name=region_name)
    table = dynamodb.Table(table_name)


    if index_name is not None:
        if days is None:
            raise ValueError("query時はdaysを指定してください。")

        now_jst = datetime.now(JST)
        start_time_str = (now_jst - timedelta(days=days)).strftime("%Y-%m-%dT%H:%M:%S+09:00")
        end_time_str = now_jst.strftime("%Y-%m-%dT%H:%M:%S+09:00")

        response = table.query(
            IndexName = index_name,
            KeyConditionExpression=(
                Key("user_id").eq(user_id) &
                Key("delivered_at").between(start_time_str, end_time_str)
            )
        )

        items = response.get('Items', [])
        return items

    else:
        response = table.get_item(Key={"user_id": user_id})
        return response.get("Item")

In [ ]:
env_path = os.path.join(os.getcwd(), "../.env")
load_dotenv(dotenv_path=env_path, override=True)
LINE_user_id = os.getenv("LINE_user_id")

table_name_list = [
    "childcare-info-tests-table1-deliverycontent",
    "childcare-info-tests-table2-userprofile",
    "childcare-info-tests-table3-categoryscore",
]

In [29]:
JST = timezone(timedelta(hours=9))

# Table1: delivery_id(PK)とは別に、「あるuser_idの直近N日分」を検索したいので、GSI1(PK: user_id, SK: delivered_at)を作成しました。そのためindex_nameを指定してqueryする必要があります。
deliverycontent = get_dynamo_data(table_name_list[0], LINE_user_id, index_name="GSI1", days=5)

# 以下の二つは、user_idで一意に取得できるので、index_nameは不要でOKです。
userprofile = get_dynamo_data(table_name_list[1], LINE_user_id)
categoryscore = get_dynamo_data(table_name_list[2], LINE_user_id)

In [ ]:
deliverycontent = [{"サマリー":d["summary"],"カテゴリー":d["category"]}
                   for d in deliverycontent]

def calc_age_month(birth_date):
    today= date.today()
    birth_date = date.fromisoformat(birth_date)
    date_difference = relativedelta(today, birth_date)
    years_plus_month = round(date_difference.years + date_difference.months / 12, 1)
    return years_plus_month

userprofile = {
    "価値観": userprofile["values"],
    "子供の情報":[
        {"子供の名前":c["child_name"],
         "月齢": calc_age_month(c["birth_date"])
         }
         for c in userprofile["children"]
    ]
}

CATEGORY_NAME_MAP = {
    "sleep": "睡眠",
    "food": "食事・栄養",
    "growth": "発達・成長",
    "health": "健康・体調管理",
    "safety": "安全",
    "play": "遊び・おでかけ",
    "daycare": "保育園・制度",
    "parent_care": "親のケア",
}

categories = set()
for key in categoryscore:
    if key.endswith("_sum"):
        categories.add(key[:-len("_sum")])
    elif key.endswith("_count"):
        categories.add(key[:-len("_count")])
result = {
    cat:float(categoryscore[f"{cat}_sum"]) / float(categoryscore[f"{cat}_count"])
    for cat in categories
}
categoryscore = sorted(result.items() , key=lambda x:x[1],reverse=True)[:3]
categoryscore = [{"カテゴリー":CATEGORY_NAME_MAP[cat],"スコア":score} for cat,score in categoryscore]

In [ ]:
def create_response(prompt_text,today,deliverycontent,userprofile,categoryscore):
    client = anthropic.Anthropic()
    prompt = PromptTemplate(
        input_variables=["today", "deliverycontent", "userprofile", "categoryscore"],
        template = prompt_text)
    prompt = prompt.format(today=today, deliverycontent=deliverycontent, userprofile=userprofile, categoryscore=categoryscore)

    message = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=5000,
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    tools = [
        {"type":"web_search_20260318","name":"web_search"}
    ]
    )

    answer_text = None
    for block in message.content:
        if block.type == "text":  
            answer_text = block.text

    return answer_text

def create_childhood_content(deliverycontent,userprofile,categoryscore):
    load_api_key()

    prompt_path = os.path.join(os.getcwd(), "../app/api/prompt.txt")
    with open(prompt_path, "r", encoding="utf-8") as f:
        prompt_text = f.read()
    today = datetime.now(JST).strftime("%Y-%m-%d")
    answer = create_response(prompt_text, today, deliverycontent, userprofile, categoryscore)
    return answer

In [ ]:
answer = create_childhood_content(deliverycontent, userprofile, categoryscore)
parsed = json.loads(answer)
print(json.dumps(parsed, indent=2, ensure_ascii=False))

message.content: [ThinkingBlock(signature='EvAbCpABCBEYAipApn05wS0Vha/xcF757viHT/LLPRGrAuZBBQAQSr7MSKsvaI3+opW22+KEPlVVdVUssTArTRuRqXkJhIhEIv8Y0DIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiRiNTMyZTUxMi03ZTgwLTQyZTUtYTdhZS01MTU2MTMyOGZlOGKoAdmz7dQGEgzGEABrryzZtZHPoNoaDHAKcWALfuKREton4CIwRzULmwJOqQTfz/ssM8h0QrPzRwqWstLciIikCPmJtQd8ew1cbeui6Uo0eNt7iXFAKowawg7dNNYqtXxuSEdLj4pZYdL9Xr0QpgR9++nnG66mxxLdsJ3O2XvpTwnSgfFqpm0P3+KRz9DZFJgPYqIz9Kqj9bKwBHHC+l2+wcKYrVV0iRFdPtIXkFTLLc9aIdw9mssEZbJc79lIr3PI83ETgUNpdI7Vbo+TsciovLzY/Y/ZiRjis8ILLTOFbm/wJSk6wiPAMbciX/iap6eDOFYsRv5biD0rJF2QeyEJZ739u8nzVFOZSKB6SpjKJSshRNUagJk3KQz6ZZ8hK2TYZVKs89JeoQoCNmNWoVM+8X6zFoWw0KEsDFs4OhSgJ26xJAnUO9H1UwM5RFuWEYudnZHMSx3f0mA9Q4mPL4hMkVO9/v5sSb/GtBSjURgSkk2XTNe+HLvbSW1VjG4JDEKNB8dRIB3azkCoP99UFJxycZSyAa93Coh6Feghsv3U68AmJdvs1VN30YLE+RhvvZzpUbp0/+IQlzVHeIZ7gfQh5GVMKTgxfy51aoIOovfJtUcNQMb8ZQsqC1EbNXIoGTB1VL2OOUix5LvFLkJjvxpxI1ZqplLPOadowup3B8ZI+gHiSgDAWbMFx3+ObtwZrZfo/ogohbNpNjvVCJC3yGak3XEAjDOz/BR0HIUP3sXG8cQ3fcRp31RXt6guS6Q/n

'{"children": [{"child_name": "子供1", "contents": [{"category": "安全", "body": "生後2か月前後はSIDS(乳幼児突然死症候群)のリスクがやや高まる時期と言われています。あおむけ寝にする、柔らかい布団やぬいぐるみを避けるなど、安全な睡眠環境を整えましょう。またこの時期から予防接種も始まるため、体調管理と合わせて準備しておくと安心です。", "summary": "生後2か月前後の安全な睡眠と予防接種準備"}]}, {"child_name": "子供2", "contents": [{"category": "睡眠", "body": "新生児は昼夜関係なく、短い睡眠と覚醒を繰り返すのが自然な姿。無理に長時間眠らせようとせず、赤ちゃんのペースに寄り添うことが大切です。日中は部屋を明るく、夜の授乳時は照明を落とすと、少しずつ生活リズムが整いやすくなると言われています。", "summary": "新生児の睡眠サイクルと昼夜リズムの整え方"}]}]}'